# ETL Pipeline and Database Benchmarks

In [ ]:
import duckdb, time, os, sys
sys.path.append('..')
import matplotlib.pyplot as plt
from memory_profiler import memory_usage
from pipeline.load import load_facts_batched

con = duckdb.connect('../ecommerce.duckdb')

# --- BATCH SIZE BENCHMARKS ---
batch_sizes = [10_000, 50_000, 100_000, 500_000]
results = []

for bs in batch_sizes:
    # Re-create fact table empty for clean benchmark
    con.execute('DELETE FROM fact_events')
    start = time.perf_counter()
    # Call your load function with this batch size
    metrics = load_facts_batched(con, 'clean_oct', batch_size=bs)
    elapsed = time.perf_counter() - start
    rps = round(metrics['rows_loaded'] / elapsed) if elapsed > 0 else 0
    results.append({'batch': bs, 'seconds': round(elapsed,2), 'rows_per_sec': rps})
    print(f'Batch {bs:>8,}: {elapsed:.2f}s | {rps:,} rows/sec')

# Chart: Batch size vs throughput
plt.figure(figsize=(10,5))
plt.plot([r['batch'] for r in results], [r['rows_per_sec'] for r in results],
         marker='o', linewidth=2, color='steelblue')
plt.xlabel('Batch Size (rows)')
plt.ylabel('Throughput (rows/second)')
plt.title('Batch Size vs Insert Throughput')
plt.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('../reports', exist_ok=True)
plt.savefig('../reports/batch_throughput.png', dpi=150)
plt.show()

# --- QUERY BENCHMARKS (with and without indexes) ---
queries = {
    'Q1_funnel':   'SELECT p.category_main, COUNT(*) FROM fact_events e JOIN dim_product p ON e.product_key=p.product_key GROUP BY 1 LIMIT 20',
    'Q2_session':  'SELECT user_session, COUNT(*) FROM fact_events GROUP BY user_session LIMIT 10',
    'Q3_brand':    'SELECT p.brand, SUM(e.price) FROM fact_events e JOIN dim_product p ON e.product_key=p.product_key WHERE e.event_type=\'purchase\' GROUP BY 1 LIMIT 10',
    'Q4_churn':    'SELECT COUNT(DISTINCT user_key) FROM fact_events WHERE event_month=10',
    'Q5_hourly':   'SELECT d.hour, COUNT(*) FROM fact_events e JOIN dim_date d ON e.date_key=d.date_key WHERE e.event_type=\'purchase\' GROUP BY 1 ORDER BY 1'
}

# Run without indexes
con.execute('DROP INDEX IF EXISTS idx_events_type')
con.execute('DROP INDEX IF EXISTS idx_events_user')
times_no_idx = {}
for name, sql in queries.items():
    start = time.perf_counter()
    con.execute(sql).fetchall()
    times_no_idx[name] = round(time.perf_counter() - start, 4)

# Recreate indexes
con.execute('CREATE INDEX idx_events_type ON fact_events(event_type)')
con.execute('CREATE INDEX idx_events_user ON fact_events(user_key)')
times_with_idx = {}
for name, sql in queries.items():
    start = time.perf_counter()
    con.execute(sql).fetchall()
    times_with_idx[name] = round(time.perf_counter() - start, 4)

# Chart: Query times with/without indexes
fig, ax = plt.subplots(figsize=(12,5))
x = range(len(queries))
names = list(queries.keys())
ax.bar([i-0.2 for i in x], [times_no_idx[n] for n in names],  0.4, label='Without Index', color='coral')
ax.bar([i+0.2 for i in x], [times_with_idx[n] for n in names], 0.4, label='With Index',    color='steelblue')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20)
ax.set_ylabel('Execution Time (seconds)')
ax.set_title('Query Execution Time: With vs Without Indexes')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/query_benchmark.png', dpi=150)
plt.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))